<a href="https://colab.research.google.com/github/Nicolas-Bernal/Simulacion-Estadistica/blob/main/Segundo%20Corte/03_RED_De_Colas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.5.2. Red de colas

Se estudiará un sistema de colas compuesto por tres nodos de servicio, siguiendo el modelo presentado en la sección 5.5.2 del libro *Simulación: Métodos y aplicaciones*.

Los clientes llegan al primer nodo de acuerdo con un proceso de Poisson. Después de recibir el servicio en el primer nodo, el 40 % de los clientes se dirige al segundo nodo y el 60 % pasa directamente al tercero. Los clientes atendidos en el segundo nodo continúan hacia el tercero, donde finalizan su recorrido.

El objetivo es simular el comportamiento de la red durante un tiempo determinado, teniendo en cuenta las llegadas, los tiempos de servicio, las colas y las rutas de los clientes.

Se calcularán el número medio de clientes en cada nodo, el tiempo medio que los clientes permanecen en el sistema y el tiempo total de simulación.

### Descripción del sistema

- **Nodo 1:** recibe las llegadas externas y atiende a los clientes en orden de llegada.
- **Nodo 2:** recibe el 40 % de los clientes que terminan el servicio en el nodo 1.
- **Nodo 3:** recibe el 60 % de los clientes del nodo 1 y todos los clientes que terminan el servicio en el nodo 2.

Cada nodo tiene una sola instalación de servicio y una cola FIFO (*First In, First Out*), es decir, el primero en llegar es el primero en ser atendido.

Para realizar la simulación se utilizarán distribuciones exponenciales y normales, como en el modelo descrito en el documento. Los parámetros numéricos se definirán explícitamente para poder ejecutar el ejemplo y se podrán modificar según los datos disponibles.

In [1]:
set.seed(123)

T=1000

lambda=0.8

mu1=1
sigma1=0.2

media2=0.8

mu31=1.2
sigma31=0.2

mu32=1.5
sigma32=0.3

prob_nodo2=0.4

### Inicialización del sistema

Se inicializan el reloj de simulación, las colas de los tres nodos, el estado de cada servidor y los contadores de llegadas y salidas.

También se almacenará el tiempo de llegada de cada cliente al sistema para calcular cuánto tiempo permanece en él. El horizonte de planificación será \(T=1000\) unidades de tiempo.

In [2]:
llegada=function() {
  rexp(1, rate=lambda)
}

servicio1=function() {
  max(0.01, rnorm(1, mu1, sigma1))
}

servicio2=function() {
  rexp(1, rate=1/media2)
}

servicio3=function(n) {
  if (n<5) {
    max(0.01, rnorm(1, mu31, sigma31))
  } else {
    max(0.01, rnorm(1, mu32, sigma32))
  }
}

### Simulación de eventos discretos

El sistema avanzará de un evento al siguiente. Los eventos posibles son:

1. Llegada de un cliente al nodo 1.
2. Finalización del servicio en el nodo 1.
3. Finalización del servicio en el nodo 2.
4. Finalización del servicio en el nodo 3.

En cada evento se actualizarán las colas, los servidores ocupados, el reloj y los contadores. Las llegadas externas se generarán únicamente hasta el horizonte $(T)$, pero se permitirá que los clientes que ya ingresaron terminen su recorrido.

In [3]:
colas=list(integer(), integer(), integer())
ocupado=c(FALSE, FALSE, FALSE)
n=c(0, 0, 0)

eventos=data.frame(
  tiempo=llegada(),
  tipo="llegada",
  id=1
)

clientes=data.frame(
  id=integer(),
  llegada=numeric(),
  salida=numeric()
)

area=c(0, 0, 0)
reloj=0
ultimo_id=0
llegadas=c(0, 0, 0)
salidas=c(0, 0, 0)

agregar_evento=function(t, tipo, id) {
  eventos<<-rbind(
    eventos,
    data.frame(tiempo=t, tipo=tipo, id=id)
  )
}

iniciar_servicio=function(j, id, t) {
  ocupado[j]<<-TRUE

  if (j==1) {
    s=servicio1()
  } else if (j==2) {
    s=servicio2()
  } else {
    s=servicio3(n[3])
  }

  agregar_evento(t+s, paste0("fin", j), id)
}

entrar_nodo=function(j, id, t) {
  n[j]<<-n[j]+1
  llegadas[j]<<-llegadas[j]+1

  if (!ocupado[j]) {
    iniciar_servicio(j, id, t)
  } else {
    colas[[j]]<<-c(colas[[j]], id)
  }
}

while (nrow(eventos)>0) {

  eventos=eventos[order(eventos$tiempo), ]
  ev=eventos[1, ]
  eventos=eventos[-1, ]

  t=ev$tiempo
  id=ev$id

  area=area+n*(t-reloj)
  reloj=t

  if (ev$tipo=="llegada") {

    ultimo_id=max(ultimo_id, id)

    clientes=rbind(
      clientes,
      data.frame(id=id, llegada=t, salida=NA_real_)
    )

    entrar_nodo(1, id, t)

    siguiente=t+llegada()

    if (siguiente<=T) {
      agregar_evento(
        siguiente, "llegada", id+1
      )
    }

  } else {

    j=as.integer(sub("fin", "", ev$tipo))

    n[j]=n[j]-1
    salidas[j]=salidas[j]+1
    ocupado[j]=FALSE

    if (j==1) {

      if (runif(1)<=prob_nodo2) {
        entrar_nodo(2, id, t)
      } else {
        entrar_nodo(3, id, t)
      }

    } else if (j==2) {

      entrar_nodo(3, id, t)

    } else {

      clientes$salida[clientes$id==id]=t

    }

    if (length(colas[[j]])>0) {
      prox=colas[[j]][1]
      colas[[j]]=colas[[j]][-1]
      iniciar_servicio(j, prox, t)
    }
  }
}

### Cálculo de indicadores de desempeño

Una vez finalizada la simulación, se calcularán los siguientes indicadores:

- Número medio de clientes en cada nodo.
- Tiempo medio de permanencia de los clientes en el sistema.
- Número de llegadas y salidas en cada nodo.
- Tiempo total transcurrido hasta que el último cliente abandona el sistema.
- Tiempo adicional después del horizonte de planificación, $(T_p)$.

El número medio de clientes se calculará mediante el área acumulada bajo la función que representa la cantidad de clientes presentes en cada nodo a lo largo del tiempo.

In [4]:
tiempo_medio=mean(
  clientes$salida-clientes$llegada,
  na.rm=TRUE
)

numero_medio=area/reloj

resultados=data.frame(
  Indicador=c(
    "Tiempo total de simulación",
    "Tiempo adicional Tp",
    "Tiempo medio en el sistema",
    "Número medio de clientes nodo 1",
    "Número medio de clientes nodo 2",
    "Número medio de clientes nodo 3"
  ),
  Valor=c(
    reloj,
    max(0, reloj-T),
    tiempo_medio,
    numero_medio[1],
    numero_medio[2],
    numero_medio[3]
  )
)

print(resultados)

cat("\nLlegadas por nodo:\n")
print(llegadas)

cat("\nSalidas por nodo:\n")
print(salidas)

cat("\nClientes que terminaron el recorrido:",
    sum(!is.na(clientes$salida)), "\n")

                        Indicador        Valor
1      Tiempo total de simulación 1207.6608629
2             Tiempo adicional Tp  207.6608629
3      Tiempo medio en el sistema   97.1691089
4 Número medio de clientes nodo 1    2.5684389
5 Número medio de clientes nodo 2    0.2505652
6 Número medio de clientes nodo 3   62.4345366

Llegadas por nodo:
[1] 811 320 811

Salidas por nodo:
[1] 811 320 811

Clientes que terminaron el recorrido: 811 


### Análisis de los resultados

Los resultados permiten evaluar el comportamiento de la red de colas.

- Un mayor número medio de clientes en un nodo indica que existe una mayor cantidad de clientes esperando o recibiendo atención.
- El tiempo medio en el sistema representa el tiempo transcurrido desde la llegada del cliente hasta la finalización del servicio en el nodo 3.
- Las llegadas y salidas permiten verificar el flujo de clientes entre las instalaciones.
- El tiempo adicional $(T_p)$ representa cuánto tarda el sistema en atender a los clientes que ingresaron antes de finalizar el horizonte de planificación.

Los resultados dependen de los parámetros de llegada y servicio utilizados. Para obtener estimaciones más confiables, es conveniente repetir la simulación con diferentes semillas y comparar los indicadores obtenidos.